In [ ]:
#!pip install rdflib networkx torch torch-geometric matplotlib

In [1]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.utils import from_networkx

from rdflib.plugins.sparql import parser, algebra
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

import pandas as pd
import json

/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1) Example SPARQL Triples for Contrastive Learning

In [2]:
def load_queries_from_csv(csv_path):
    df = pd.read_csv(csv_path)

    dataset = []
    for _, row in df.iterrows():
        anchor = row["anchor_query"]

        # parse JSON list stored as text
        positives = json.loads(row["positive_queries"])
        negatives = json.loads(row["negative_queries"])

        dataset.append({
            "id": row["id"],
            "anchor": anchor,
            "positives": positives,
            "negatives": negatives
        })

    return dataset

# Example:
dataset = load_queries_from_csv("sample_sparql_queries.csv")
dataset=dataset[:1000]
print(dataset[0])

{'id': 0, 'anchor': '\nPREFIX dbo: <http://dbpedia.org/ontology/>\nSELECT ?x WHERE {\n    ?x dbo:deathPlace dbo:France .\n}\n', 'positives': ['\nPREFIX dbo: <http://dbpedia.org/ontology/>\nSELECT ?person WHERE {\n    ?person dbo:deathPlace dbo:France ;\n}\n'], 'negatives': ['\nPREFIX dbo: <http://dbpedia.org/ontology/>\nSELECT ?y WHERE {\n    {\n        ?y dbo:birthPlace dbo:Berlin .\n        OPTIONAL { ?y dbo:occupation ?job . }\n    }\n    UNION\n    {\n        ?y dbo:birthPlace dbo:London .\n    }\n}\n', '\nPREFIX dbo: <http://dbpedia.org/ontology/>\nSELECT ?z WHERE {\n    ?z dbo:occupation dbo:Berlin .\n    ?z dbo:team dbo:Berlin .\n}\n']}


 ------------------------------------------------------------
### 2) Convert SPARQL Algebra → NetworkX Graph
 ------------------------------------------------------------


In [4]:
def algebra_to_graph(op, G=None, parent=None, node_id=0):
    """
    Recursively builds a directed graph from the SPARQL algebra tree.
    Each algebra operator becomes a node, and BGP triples become leaf nodes.
    """
    if G is None:
        G = nx.DiGraph()

    # Label of the current algebra operator
    node_label = op.name if hasattr(op, 'name') else type(op).__name__
    node_name = f"{node_label}_{node_id}"

    G.add_node(node_name, label=node_label)

    if parent is not None:
        G.add_edge(parent, node_name)

    # Special case: Basic Graph Pattern (BGP) triples
    if getattr(op, "name", None) == "BGP":
        for s, p, o in op.triples:
            triple_node = f"Triple_{s}_{p}_{o}_{node_id}"
            triple_label = f"{s} {p} {o}"
            G.add_node(triple_node, label=triple_label)
            G.add_edge(node_name, triple_node)

    # Recurse over attributes/values in the algebra object
    # to find nested algebra operations
    for v in op.values():
        if hasattr(v, "name"):  # nested algebra op
            node_id += 1
            G = algebra_to_graph(v, G, node_name, node_id)
        elif isinstance(v, list):
            for sub in v:
                if hasattr(sub, "name"):
                    node_id += 1
                    G = algebra_to_graph(sub, G, node_name, node_id)

    return G


### 3) Convert NetworkX → PyTorch Geometric Data

In [5]:
def graph_to_pyg(G):
    labels = nx.get_node_attributes(G, "label")

    # Simple label hashing → 3D feature vector per node
    feats = []
    for lbl in labels.values():
        h = abs(hash(lbl)) % (10**6)
        vec = np.array([
            h % 97,
            (h // 97) % 97,
            (h // (97**2)) % 97
        ], dtype=float)
        feats.append(vec / 100.0)

    feats = np.stack(feats, axis=0)
    feats = torch.tensor(feats, dtype=torch.float)

    data = from_networkx(G)
    # Ensure we have node features in correct order
    data.x = feats
    # Single-graph batch index (all zeros)
    data.batch = torch.zeros(data.num_nodes, dtype=torch.long)

    return data


### 4) Wrapper: SPARQL Query → PyG Graph

In [6]:
def encode_sparql_query_to_graph(query_str):
    # Parse and translate to algebra
    parsed = parser.parseQuery(query_str)
    alg = algebra.translateQuery(parsed).algebra

    # Algebra → NetworkX
    G = algebra_to_graph(alg)

    # NetworkX → PyG
    pyg_data = graph_to_pyg(G)
    return pyg_data, G



### 6) GCN Encoder

In [7]:
class GCNEncoder(torch.nn.Module):
    def __init__(self, in_dim=3, hidden=32, out_dim=16):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, out_dim)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        # aggregate node embeddings to graph embedding
        x = global_mean_pool(x, batch)
        return x  # shape: [batch_size, out_dim]


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GCNEncoder().to(device)
print(model)

# ------------------------------------------------------------
# 7) Helper: Query → Embedding (with current model)
# ------------------------------------------------------------

def get_embedding(model, query_str):
    data, _ = encode_sparql_query_to_graph(query_str)
    data = data.to(device)
    model.eval()
    with torch.no_grad():
        emb = model(data.x, data.edge_index, data.batch)
    return emb  # shape [1, out_dim]


GCNEncoder(
  (conv1): GCNConv(3, 32)
  (conv2): GCNConv(32, 16)
)


### 8) InfoNCE Contrastive Loss

In [8]:
def info_nce(anchor, positives, negatives, temperature=0.1):
    """
    anchor:     [1, d]
    positives:  list of [1, d]
    negatives:  list of [1, d]
    """
    # Normalize
    anchor = anchor / anchor.norm(dim=-1, keepdim=True)
    pos = torch.cat(positives, dim=0)  # [P, d]
    neg = torch.cat(negatives, dim=0)  # [N, d]

    pos = pos / pos.norm(dim=-1, keepdim=True)
    neg = neg / neg.norm(dim=-1, keepdim=True)

    # Similarity to anchor (dot product)
    pos_sim = (pos @ anchor.T) / temperature   # [P, 1]
    neg_sim = (neg @ anchor.T) / temperature   # [N, 1]

    # Combine logits: first P are positives, then N negatives
    logits = torch.cat([pos_sim, neg_sim], dim=0).T  # [1, P+N]
    # Label index of the correct class = 0..P-1 (we treat "any positive" as target 0; simplest demo)
    # For a simple demo, we mark one representative positive as the target.
    target = torch.tensor([0], dtype=torch.long, device=logits.device)

    loss = F.cross_entropy(logits, target)
    return loss, logits.detach().cpu()


### 10) Mini Contrastive Training Loop

In [ ]:
graph_cache = {}

def get_graph_cached(query):
    if query not in graph_cache:
        pyg, _ = encode_sparql_query_to_graph(query)
        graph_cache[query] = pyg
    return graph_cache[query]

In [9]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs=20

for epoch in range(1, num_epochs + 1):
    total_loss = 0

    for item in dataset:
        anchor_query = item["anchor"]
        positive_queries = item["positives"]
        negative_queries = item["negatives"]

        # Encode data
        data_anchor, _ = encode_sparql_query_to_graph(anchor_query)
        data_anchor = data_anchor.to(device)

        pos_data = [get_graph_cached(q).to(device)
                    for q in positive_queries]
        neg_data = [get_graph_cached(q).to(device)
                    for q in negative_queries]

        model.train()
        optimizer.zero_grad()

        anchor_emb = model(data_anchor.x, data_anchor.edge_index, data_anchor.batch)
        pos_embs = [model(d.x, d.edge_index, d.batch) for d in pos_data]
        neg_embs = [model(d.x, d.edge_index, d.batch) for d in neg_data]

        loss, _ = info_nce(anchor_emb, pos_embs, neg_embs)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch} | Avg Loss: {total_loss / len(dataset):.4f}")



Epoch 1 | Avg Loss: 0.2653
Epoch 2 | Avg Loss: 0.0540
Epoch 3 | Avg Loss: 0.2675
Epoch 4 | Avg Loss: 0.0001
Epoch 5 | Avg Loss: 0.0001
Epoch 6 | Avg Loss: 0.6050
Epoch 7 | Avg Loss: 0.0029
Epoch 8 | Avg Loss: 0.0000
Epoch 9 | Avg Loss: 0.0000
Epoch 10 | Avg Loss: 0.0111
Epoch 11 | Avg Loss: 0.0004
Epoch 12 | Avg Loss: 0.0000
Epoch 13 | Avg Loss: 0.0000
Epoch 14 | Avg Loss: 0.0000
Epoch 15 | Avg Loss: 0.0000
Epoch 16 | Avg Loss: 0.0000
Epoch 17 | Avg Loss: 0.0150
Epoch 18 | Avg Loss: 0.0000
Epoch 19 | Avg Loss: 0.0000
Epoch 20 | Avg Loss: 0.0000


### 11) Inspect Similarities After Training

In [10]:
model.eval()

print("=== Cosine Similarity Test on 10 Examples ===")

for idx in range(10):
    example = dataset[idx]

    anchor_query     = example["anchor"]
    positive_queries = example["positives"]
    negative_queries = example["negatives"]

    # Compute embeddings
    anchor_emb = get_embedding(model, anchor_query)
    pos_embs   = [get_embedding(model, q) for q in positive_queries]
    neg_embs   = [get_embedding(model, q) for q in negative_queries]

    # Cosine function
    def cosine(a, b):
        a = a / a.norm(dim=-1, keepdim=True)
        b = b / b.norm(dim=-1, keepdim=True)
        return float(a @ b.T)

    print(f"\n--- Example {idx} ---")
    print("Anchor Query:")
    print(anchor_query.strip())

    print("\nPositive similarities:")
    for i, e in enumerate(pos_embs):
        print(f"  Positive {i}: {cosine(anchor_emb, e):.4f}")

    print("\nNegative similarities:")
    for j, e in enumerate(neg_embs):
        print(f"  Negative {j}: {cosine(anchor_emb, e):.4f}")


=== Cosine Similarity Test on 10 Examples ===

--- Example 0 ---
Anchor Query:
PREFIX dbo: <http://dbpedia.org/ontology/>
SELECT ?x WHERE {
    ?x dbo:deathPlace dbo:France .
}

Positive similarities:
  Positive 0: 0.9349

Negative similarities:
  Negative 0: -0.8389
  Negative 1: -0.8367

--- Example 1 ---
Anchor Query:
PREFIX dbo: <http://dbpedia.org/ontology/>
SELECT ?x WHERE {
    ?x dbo:occupation dbo:Berlin .
}

Positive similarities:
  Positive 0: 0.9399

Negative similarities:
  Negative 0: -0.8921
  Negative 1: -0.9068

--- Example 2 ---
Anchor Query:
PREFIX dbo: <http://dbpedia.org/ontology/>
SELECT ?x WHERE {
    ?x dbo:deathPlace dbo:Berlin .
}

Positive similarities:
  Positive 0: 0.9905

Negative similarities:
  Negative 0: -0.8573
  Negative 1: -0.8474

--- Example 3 ---
Anchor Query:
PREFIX dbo: <http://dbpedia.org/ontology/>
SELECT ?x WHERE {
    ?x dbo:director dbo:NewYork .
}

Positive similarities:
  Positive 0: 0.9996

Negative similarities:
  Negative 0: -0.9108
 